# 🍱 한끼 칼로리 트래커 — 음식 이미지 분류 모델 학습

**MobileNetV2 Transfer Learning on Food-101**

### 개요
- **데이터셋**: Food-101 (101 클래스 × 1,000장 = 101,000장)
- **모델**: MobileNetV2 (ImageNet pretrained) + Custom Head
- **출력 클래스**: nutrition_db.json의 30개 한국 음식 클래스
- **저장**: `food_classifier.h5` + `class_indices.json`

### ⚠️ 매핑 주의사항
Food-101에는 대부분의 한국 음식이 없습니다. 따라서 가장 유사한 Food-101 클래스를 매핑하여 사용합니다.  
예) `ramen` → `ramyeon`, `pork_chop` → `samgyeopsal`  
실제 한국 음식 이미지로 파인튜닝하면 성능이 크게 향상됩니다.

### Colab 설정
런타임 → 런타임 유형 변경 → **GPU** (T4 권장)

In [ ]:
# ── 1. 패키지 설치 및 Google Drive 마운트 ──────────────────────────────
!pip install -q tensorflow tensorflow-datasets Pillow

from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/meal-tracker-model/'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ 모델 저장 경로: {SAVE_DIR}')

In [ ]:
# ── 2. 라이브러리 임포트 ───────────────────────────────────────────────
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── 3. 클래스 매핑 정의 ────────────────────────────────────────────────
# Food-101 클래스명 → nutrition_db.json DB key (30개 한국 음식)

FOOD101_TO_DB = {
    # 직접 매칭 (3)
    'bibimbap':          'bibimbap',
    'fried_rice':        'fried_rice',
    'ramen':             'ramyeon',

    # 계란류 (5)
    'omelette':          'gyeran_mari',       # 계란말이
    'eggs_benedict':     'gyeran_bap',        # 계란밥
    'deviled_eggs':      'gyeran_jjim',       # 계란찜
    'french_toast':      'gyeran_gui',        # 계란후라이
    'huevos_rancheros':  'omurice',           # 오므라이스

    # 국/찌개 (5)
    'miso_soup':         'doenjang_jjigae',   # 된장찌개
    'hot_and_sour_soup': 'kimchi_jjigae',     # 김치찌개
    'clam_chowder':      'sundubu_jjigae',    # 순두부찌개
    'lobster_bisque':    'seolleongtang',     # 설렁탕
    'seaweed_salad':     'miyeok_guk',        # 미역국

    # 나물/채소 (5)
    'greek_salad':       'spinach_namul',     # 시금치나물
    'edamame':           'kongnamul',         # 콩나물
    'beet_salad':        'kkakdugi',          # 깍두기
    'nachos':            'japchae_vegetables',# 숙주나물
    'sushi':             'kimchi',            # 김치

    # 육류 (5)
    'steak':             'bulgogi',           # 불고기
    'pork_chop':         'samgyeopsal',       # 삼겹살
    'chicken_wings':     'dakgalbi',          # 닭갈비
    'prime_rib':         'galbi',             # 갈비
    'pho':               'jeyuk_bokkeum',     # 제육볶음

    # 해산물 (5)
    'fried_calamari':    'ojingeo_bokkeum',   # 오징어볶음
    'shrimp_and_grits':  'saewoo_gui',        # 새우구이
    'grilled_salmon':    'godeungeo_gui',     # 고등어구이
    'takoyaki':          'jeon_haemul',       # 낙지볶음
    'spring_rolls':      'haemul_pajeon',     # 해물파전

    # 밥/면류 추가 (2)
    'gyoza':             'japchae',           # 잡채
    'risotto':           'rice',              # 쌀밥
}

# DB 클래스 인덱스 (알파벳 정렬 → 일관된 인덱스 보장)
DB_CLASSES = sorted(set(FOOD101_TO_DB.values()))
DB_CLASS_TO_IDX = {cls: i for i, cls in enumerate(DB_CLASSES)}
IDX_TO_DB_CLASS = {i: cls for i, cls in enumerate(DB_CLASSES)}
NUM_CLASSES = len(DB_CLASSES)

print(f'총 클래스 수: {NUM_CLASSES}')
print('\n인덱스 → 클래스:')
for i, cls in enumerate(DB_CLASSES):
    print(f'  [{i:2d}] {cls}')

In [ ]:
# ── 4. Food-101 데이터셋 다운로드 및 라벨 매핑 ─────────────────────────
# ⚠️ 약 4.65 GB 다운로드 — 시간이 걸립니다

builder = tfds.builder('food101')
builder.download_and_prepare()
food101_names = builder.info.features['label'].names  # 101개 클래스명 리스트

name_to_f101_label = {name: i for i, name in enumerate(food101_names)}

# Food-101 label(int) → DB label(int) 매핑 테이블 생성
LABEL_REMAP = {}
print('\n매핑 현황:')
for f101_name, db_name in FOOD101_TO_DB.items():
    f101_idx = name_to_f101_label.get(f101_name)
    if f101_idx is not None:
        db_idx = DB_CLASS_TO_IDX[db_name]
        LABEL_REMAP[f101_idx] = db_idx
        print(f'  Food-101[{f101_idx:3d}] {f101_name:<22} → DB[{db_idx:2d}] {db_name}')
    else:
        print(f'  ⚠️ {f101_name} not found in Food-101')

VALID_LABELS = sorted(LABEL_REMAP.keys())
print(f'\n유효 Food-101 라벨 수: {len(VALID_LABELS)} / {len(FOOD101_TO_DB)}')

In [ ]:
# ── 5. tf.data 파이프라인 구성 ─────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

# StaticHashTable: Food-101 label → DB label
keys_t   = tf.constant(VALID_LABELS, dtype=tf.int64)
values_t = tf.constant([LABEL_REMAP[k] for k in VALID_LABELS], dtype=tf.int32)
table    = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(keys_t, values_t),
    default_value=-1
)

def preprocess(example, augment=False):
    img = tf.cast(tf.image.resize(example['image'], [IMG_SIZE, IMG_SIZE]), tf.float32)
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, max_delta=30)
        img = tf.image.random_contrast(img, 0.8, 1.2)
        img = tf.image.random_saturation(img, 0.8, 1.2)
        img = tf.image.random_hue(img, 0.05)
        img = tf.clip_by_value(img, 0.0, 255.0)
    img   = tf.keras.applications.mobilenet_v2.preprocess_input(img)  # [0,255] → [-1,1]
    label = table.lookup(tf.cast(example['label'], tf.int64))
    return img, label

def filter_valid(img, label):
    return tf.not_equal(label, -1)

# 원본 데이터셋 로드
ds_train_raw = builder.as_dataset(split='train',      shuffle_files=True)
ds_val_raw   = builder.as_dataset(split='validation', shuffle_files=False)

# 학습 파이프라인 (Augmentation 포함)
ds_train = (
    ds_train_raw
    .map(lambda x: preprocess(x, augment=True), num_parallel_calls=AUTOTUNE)
    .filter(filter_valid)
    .cache()
    .shuffle(2000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# 검증 파이프라인 (Augmentation 없음)
ds_val = (
    ds_val_raw
    .map(lambda x: preprocess(x, augment=False), num_parallel_calls=AUTOTUNE)
    .filter(filter_valid)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f'예상 학습 샘플: ~{len(VALID_LABELS) * 750:,}개 (클래스당 ~750개)')
print(f'예상 검증 샘플: ~{len(VALID_LABELS) * 250:,}개 (클래스당 ~250개)')
print(f'배치 크기: {BATCH_SIZE}')

In [ ]:
# ── 6. 샘플 이미지 시각화 ──────────────────────────────────────────────
sample_imgs, sample_labels = next(iter(ds_train))

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i, ax in enumerate(axes.flatten()):
    if i >= len(sample_imgs): break
    img = (sample_imgs[i].numpy() + 1.0) / 2.0  # [-1,1] → [0,1]
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(IDX_TO_DB_CLASS[int(sample_labels[i])], fontsize=7)
    ax.axis('off')
plt.suptitle('학습 데이터 샘플 (Data Augmentation 적용)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7. MobileNetV2 + Custom Head 모델 구성 ────────────────────────────
def build_model(num_classes=NUM_CLASSES):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False  # Phase 1: 동결

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs, name='food_classifier')
    return model, base

model, base_model = build_model()
model.summary()

total_params     = model.count_params()
trainable_params = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'\n전체 파라미터: {total_params:,}')
print(f'학습 가능 파라미터: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)')

In [ ]:
# ── 8. Phase 1: Custom Head 학습 (MobileNetV2 완전 동결) ───────────────
PHASE1_EPOCHS = 15

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_phase1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=2, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(SAVE_DIR, 'ckpt_phase1.keras'),
        save_best_only=True, monitor='val_accuracy', verbose=0),
]

print('=== Phase 1: Custom Head 학습 ===')
history1 = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=PHASE1_EPOCHS,
    callbacks=cb_phase1
)

val_acc1 = max(history1.history['val_accuracy'])
print(f'\nPhase 1 최고 검증 정확도: {val_acc1:.4f}')

In [ ]:
# ── 9. Phase 1 학습 결과 시각화 ───────────────────────────────────────
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(history.history['accuracy'],     label='train')
    ax1.plot(history.history['val_accuracy'], label='val')
    ax1.set_title(f'{title} — Accuracy')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(history.history['loss'],     label='train')
    ax2.plot(history.history['val_loss'], label='val')
    ax2.set_title(f'{title} — Loss')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

plot_history(history1, 'Phase 1 (Frozen MobileNetV2)')

In [ ]:
# ── 10. Phase 2: Fine-tuning (MobileNetV2 마지막 30 레이어 언프리즈) ────
FINETUNE_LAYERS = 30
PHASE2_EPOCHS   = 25

base_model.trainable = True
for layer in base_model.layers[:-FINETUNE_LAYERS]:
    layer.trainable = False

trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Phase 2 학습 가능 파라미터: {trainable:,}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),  # 낮은 LR으로 파인튜닝
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_phase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=3, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(SAVE_DIR, 'ckpt_phase2.keras'),
        save_best_only=True, monitor='val_accuracy', verbose=0),
]

print('=== Phase 2: Fine-tuning ===')
history2 = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=PHASE2_EPOCHS,
    callbacks=cb_phase2,
    initial_epoch=len(history1.history['accuracy'])
)

val_acc2 = max(history2.history['val_accuracy'])
print(f'\nPhase 2 최고 검증 정확도: {val_acc2:.4f}')

In [ ]:
# ── 11. Phase 2 결과 시각화 ───────────────────────────────────────────
plot_history(history2, 'Phase 2 (Fine-tuning)')

In [ ]:
# ── 12. 최종 평가 ─────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
!pip install -q seaborn

val_loss, val_acc = model.evaluate(ds_val, verbose=1)
print(f'\n최종 검증 정확도: {val_acc:.4f} ({val_acc*100:.1f}%)')
print(f'최종 검증 손실:   {val_loss:.4f}')

# 예측 수집
y_true, y_pred = [], []
for imgs, labels in ds_val:
    preds = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print('\n분류 리포트:')
print(classification_report(y_true, y_pred, target_names=DB_CLASSES))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=DB_CLASSES, yticklabels=DB_CLASSES)
plt.title('Confusion Matrix')
plt.ylabel('실제'); plt.xlabel('예측')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── 13. 모델 저장 ─────────────────────────────────────────────────────
MODEL_H5_PATH   = os.path.join(SAVE_DIR, 'food_classifier.h5')
CLASS_IDX_PATH  = os.path.join(SAVE_DIR, 'class_indices.json')

# H5 형식으로 저장 (backend/app/ 에 복사하면 바로 사용 가능)
model.save(MODEL_H5_PATH)
print(f'✅ 모델 저장: {MODEL_H5_PATH}')

# 클래스 인덱스 저장 (index → DB key 매핑)
class_indices = {str(i): cls for i, cls in IDX_TO_DB_CLASS.items()}
with open(CLASS_IDX_PATH, 'w', encoding='utf-8') as f:
    json.dump(class_indices, f, ensure_ascii=False, indent=2)
print(f'✅ 클래스 인덱스 저장: {CLASS_IDX_PATH}')

print('\n클래스 인덱스 목록:')
for idx, cls in sorted(class_indices.items(), key=lambda x: int(x[0])):
    print(f'  [{idx:>2}] {cls}')

print('\n' + '='*50)
print('📌 다음 단계:')
print('  1. 구글 드라이브에서 아래 두 파일 다운로드:')
print(f'     - {MODEL_H5_PATH}')
print(f'     - {CLASS_IDX_PATH}')
print('  2. 두 파일을 backend/app/ 폴더에 복사')
print('  3. pip install tensorflow Pillow numpy')
print('  4. 백엔드 서버 재시작')
print('  5. GET /api/health → ml_model_loaded: true 확인')
print('='*50)

In [ ]:
# ── 14. 추론 테스트 (검증 데이터 샘플) ───────────────────────────────
test_imgs, test_labels = next(iter(ds_val))
preds = model.predict(test_imgs[:16], verbose=0)

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i, ax in enumerate(axes.flatten()):
    img = (test_imgs[i].numpy() + 1.0) / 2.0
    ax.imshow(np.clip(img, 0, 1))
    true_cls = IDX_TO_DB_CLASS[int(test_labels[i])]
    pred_cls = IDX_TO_DB_CLASS[int(np.argmax(preds[i]))]
    conf     = float(np.max(preds[i]))
    color    = 'green' if true_cls == pred_cls else 'red'
    ax.set_title(f'{pred_cls}\n({conf:.0%})', color=color, fontsize=7)
    ax.axis('off')

plt.suptitle('추론 테스트 (초록=정답, 빨강=오답)', fontsize=12)
plt.tight_layout()
plt.show()